In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, f_oneway
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score
import logging
from diffprivlib.models import GaussianNB  # Simulated differential privacy
from joblib import dump
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

In [7]:
# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [9]:
# Privacy-awareData Loading
def load_data_privacy_aware(file_path):
    logging.info("Loading data with privacy simulation")
    df = pd.read_csv(file_path)
    # Simulate differential privacy noise on numeric columns
    numeric_cols = ['Views', 'Likes', 'Shares', 'Comments']
    for col in numeric_cols:
        noise = np.random.laplace(0, 0.1, size=len(df))  # Small noise for demo
        df[col] = df[col] + noise
        df[col] = df[col].clip(lower=0)  # Ensure non-negative
    return df

In [10]:
# Data Pipeline
def process_data(df):
    logging.info("Processing data in pipeline")
    df.dropna(inplace=True)
    for col in ['Platform', 'Hashtag', 'Content_Type', 'Region', 'Engagement_Level']:
        df[col] = df[col].astype('category')
    numeric_cols = ['Views', 'Likes', 'Shares', 'Comments']
    scaler = StandardScaler()
    df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
    df['Engagement_Score'] = df[numeric_cols].mean(axis=1)
    df['Log_Views'] = np.log1p(df['Views'].clip(lower=0))
    return df, scaler